In [1]:
import pandas as pd
import numpy as np
from lifetimes.utils import summary_data_from_transaction_data
from lifetimes import BetaGeoFitter
from lifetimes import GammaGammaFitter

# Load data
df = pd.read_csv("group_assignment_2026.csv")

# Fit `lifetimes` model to retreive predictive CLV:

For the benchmark CLV model, we simply only pay attention to repeated customers' past purchases to the brand in general, and predict their CLV into the future.

CLV here is represented by the total revenue that a customer generates (= tot. `PriceExcVat`).

## Pre-prossessing

Standardize timestamp and clean instances where revenue/monetary value (PirceIncVAT/PirceExcVAT <= 0)

In [2]:
# Drop timezone information:
df["DeliveryTimestamp"] = pd.to_datetime(df["DeliveryTimestamp"]).dt.tz_localize(None)

# Drop 0 value orders. In other words, drop instances where monetary values = 0:
df = df[df['PriceExcVat'] > 0]
df = df[df['PriceIncVat'] > 0]

Apply the package `lifetimes`

In [3]:
observation_end = df['DeliveryTimestamp'].max()

lifetimes_summary = summary_data_from_transaction_data(
    df,
    customer_id_col='CustomerID',
    datetime_col='DeliveryTimestamp',
    monetary_value_col='PriceExcVat',
    observation_period_end=observation_end
)
lifetimes_summary['frequency'].value_counts().head(10)

frequency
0.0    549274
1.0    138694
2.0     64327
3.0     36191
4.0     22810
5.0     15695
6.0     10817
7.0      7998
8.0      6023
9.0      4430
Name: count, dtype: int64

In [4]:
# total customers
total_customers = len(lifetimes_summary)

# customers with frequency = 0
zero_freq_customers = (lifetimes_summary['frequency'] == 0).sum()

# percentage
zero_freq_pct = zero_freq_customers / total_customers * 100

print(f"Customers with frequency = 0: {zero_freq_customers}")
print(f"Percentage of total: {zero_freq_pct:.2f}%")

Customers with frequency = 0: 549274
Percentage of total: 62.73%


## BG/NBD purchase model

Run the purchase model based on the whole dataset

In [5]:
bgf = BetaGeoFitter(penalizer_coef=0.001)

bgf.fit(
    lifetimes_summary['frequency'],
    lifetimes_summary['recency'],
    lifetimes_summary['T']
)
bgf.summary


,coef,se(coef),lower 95% bound,upper 95% bound
r,0.239791,0.000644,0.238529,0.241053
alpha,53.692469,0.227117,53.247320,54.137618
a,0.440015,0.003104,0.433931,0.446099
b,1.511156,0.012402,1.486849,1.535464


In [7]:
lifetimes_summary['pred_purchases_6m'] = bgf.predict(
    180,
    lifetimes_summary['frequency'],
    lifetimes_summary['recency'],
    lifetimes_summary['T']
)

In [8]:
# Sanity-check predicted purchases (180 days ≈ 6 months)
print("=== pred_purchases_6m descriptive stats ===\n")
print(lifetimes_summary['pred_purchases_6m'].describe())

print(f"\nCustomers with 0 predicted purchases: "
      f"{(lifetimes_summary['pred_purchases_6m'] == 0).sum()}")

print(f"Customers with >10 predicted purchases: "
      f"{(lifetimes_summary['pred_purchases_6m'] > 10).sum()}")

# Cross-check: do customers with higher historical frequency 
# get higher predicted purchases? (should be positive)
print(f"\nCorrelation(frequency, pred_purchases_6m): "
      f"{lifetimes_summary['frequency'].corr(lifetimes_summary['pred_purchases_6m']):.3f}")

# Compare mean predicted purchases against naive historical rate
# (avg purchases per 180 days based on observed frequency/T)
repeaters = lifetimes_summary[lifetimes_summary['frequency'] > 0]
naive_rate = (repeaters['frequency'] / repeaters['T'] * 180).mean()
model_rate = repeaters['pred_purchases_6m'].mean()
print(f"\nRepeaters only:")
print(f"  Naive historical rate (freq/T * 180): {naive_rate:.2f}")
print(f"  BG/NBD predicted mean:                {model_rate:.2f}")

=== pred_purchases_6m descriptive stats ===

count    8.755580e+05
mean     3.594690e-01
std      7.185861e-01
min      6.250929e-75
25%      7.609485e-02
50%      1.331514e-01
75%      3.340218e-01
max      6.412860e+01
Name: pred_purchases_6m, dtype: float64

Customers with 0 predicted purchases: 0
Customers with >10 predicted purchases: 387

Correlation(frequency, pred_purchases_6m): 0.782

Repeaters only:
  Naive historical rate (freq/T * 180): 1.35
  BG/NBD predicted mean:                0.73


In [9]:
repeaters = lifetimes_summary[lifetimes_summary['frequency'] > 0]
repeaters.shape


(326284, 5)

In [10]:
repeaters[['frequency','monetary_value']].corr()

,frequency,monetary_value
frequency,1.000000,-0.077977
monetary_value,-0.077977,1.000000


In [11]:
ggf = GammaGammaFitter(penalizer_coef=0.001)

ggf.fit(
    repeaters['frequency'],
    repeaters['monetary_value']
)
ggf.summary


,coef,se(coef),lower 95% bound,upper 95% bound
p,11.676017,0.024092,11.628797,11.723238
q,3.664829,0.008848,3.647487,3.682170
v,10.634606,0.023847,10.587867,10.681346


Fit CLV predictions for all

In [13]:
lifetimes_summary['CLV_6m'] = ggf.customer_lifetime_value(
    bgf,
    lifetimes_summary['frequency'],
    lifetimes_summary['recency'],
    lifetimes_summary['T'],
    lifetimes_summary['monetary_value'],
    time=6,
    discount_rate=0.01
)

lifetimes_summary['CLV_6m'].describe()


count    8.755580e+05
mean     1.465119e+01
std      2.791636e+01
min      4.962536e-73
25%      3.366665e+00
50%      5.679045e+00
75%      1.415012e+01
max      2.803953e+03
Name: CLV_6m, dtype: float64

In [14]:
lifetimes_summary = lifetimes_summary.reset_index()

customer_info = df.groupby('CustomerID').agg({
    'Channel': lambda x: x.mode()[0],
    'StoreName': lambda x: x.mode()[0]
}).reset_index()

lifetimes_summary = lifetimes_summary.merge(
    customer_info,
    on='CustomerID',
    how='left'
)

In [15]:
try:
    # Try running aggregation directly
    result = lifetimes_summary.groupby('Channel')['CLV_6m'].agg(['mean', 'median', 'count'])

except Exception as e:
    print("Error detected, fixing duplicate columns...")

    # Fix duplicated columns ONLY if error occurs
    lifetimes_summary = lifetimes_summary.rename(columns={
        'Channel_x': 'Channel',
        'StoreName_x': 'StoreName'
    })

    lifetimes_summary = lifetimes_summary.drop(
        columns=['Channel_y', 'StoreName_y'],
        errors='ignore'
    )

    # Retry aggregation
    result = lifetimes_summary.groupby('Channel')['CLV_6m'].agg(['mean', 'median', 'count'])

# Display result
result

,mean,median,count
Channel,,,
ConsumerApp,7.186103,4.092245,9294
Intake module,3.712196,3.147925,16589
S4D - Cash register,2.422052,2.470501,6
TakeAway,14.932903,6.053212,609170
UberEats,4.305305,3.645399,17014
Website,15.793697,5.851241,223485


In [16]:
lifetimes_summary.groupby('StoreName')['CLV_6m'].mean().sort_values(ascending=False)

StoreName
Spare Rib Express Heerlen            36.443106
Spare Rib Express Leidschendam       20.871507
Spare Rib Express Middelburg         18.822357
Spare Rib Express Geldermalsen       18.618333
Spare Rib Express Den Helder         17.974841
                                       ...    
Spare Rib Express Almere Poort        3.990651
Spare Rib Express Amsterdam Noord     3.853880
Spare Rib Express Oosterhout          3.519672
Spare Rib Express Almelo              3.479324
Spare Rib Express Hengelo             2.849515
Name: CLV_6m, Length: 82, dtype: float64

In [17]:
# Export full dataframe with CLV_6m for better workflow on other devices:
lifetimes_summary.to_csv("lifetimes_summary_with_CLV.csv", index=False)

# Figures

All figures are saved to the `figures/` sub-folder.

In [18]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import os

OUTDIR = "figures"
os.makedirs(OUTDIR, exist_ok=True)

plt.rcParams.update({
    "font.family":      "Liberation Sans",
    "font.size":        11,
    "axes.spines.top":  False,
    "axes.spines.right": False,
    "axes.titlesize":   12,
    "axes.titleweight": "bold",
    "figure.dpi":       150,
})

BLUE   = "#2C7BB6"
RED    = "#D7191C"
ORANGE = "#FDAE61"

def save(name):
    path = os.path.join(OUTDIR, name)
    plt.savefig(path, bbox_inches="tight", dpi=150)
    plt.close()
    print(f"Saved: {path}")


## Figure 1: Customer Purchase Frequency Distribution

In [19]:
CAP   = 10
total = len(lifetimes_summary)
freq_capped = lifetimes_summary["frequency"].clip(upper=CAP)
counts = freq_capped.value_counts().sort_index()
labels = [str(int(x)) if x < CAP else f"{CAP}+" for x in counts.index]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(labels, counts.values, color=BLUE, edgecolor="white", linewidth=0.5, width=0.7)

for bar, cnt in zip(bars, counts.values):
    pct = cnt / total * 100
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + total * 0.003,
            f"{pct:.1f}%", ha="center", va="bottom", fontsize=8.5, color="#333333")

ax.set_xlabel("Number of repeat purchases (frequency)")
ax.set_ylabel("Number of customers")
ax.set_title("Figure 1: Distribution of Customer Purchase Frequency")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.set_ylim(0, counts.max() * 1.13)
fig.tight_layout()
save("figure_1_frequency_distribution.png")


findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberati

Saved: figures\figure_1_frequency_distribution.png


In [20]:
# Build customer-level historical summary for Figures 2 & 3
clv_table = df.groupby('CustomerID').agg(
    revenue=('PriceExcVat', 'sum'),
    first_purchase=('DeliveryTimestamp', 'min'),
    last_purchase=('DeliveryTimestamp', 'max')
).reset_index()

## Figure 2: Historical Revenue per Customer

In [21]:
# clv_table was built at the top of this notebook from the raw data
rev      = clv_table["revenue"]
cap      = rev.quantile(0.99)
mean_rev = rev.mean()

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(rev[rev <= cap], bins=60, color=BLUE, edgecolor="white", linewidth=0.3)
ax.axvline(mean_rev, color=RED, linewidth=1.8, linestyle="--", label=f"Mean: €{mean_rev:.0f}")
ax.set_xlabel("Historical revenue per customer (€ excl. VAT, top 1% trimmed)")
ax.set_ylabel("Number of customers")
ax.set_title("Figure 2: Distribution of Historical Revenue per Customer")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"€{int(x):,}"))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.legend(frameon=False)
fig.tight_layout()
save("figure_2_historical_revenue.png")

findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberati

Saved: figures\figure_2_historical_revenue.png


## Figure 3: Historical Customer Lifetime Distribution

Because ~63% of customers are one-timers with lifetime = 0 days, a single histogram is dominated by the spike at zero and hides the shape of the repeat-buyer population. We therefore use two panels: the left shows the full picture; the right zooms into repeat buyers only (lifetime > 0) to reveal the actual spread.

In [22]:
# Add lifetime_days to clv_table if not already present
clv_table["first_purchase"] = pd.to_datetime(clv_table["first_purchase"], utc=True, errors="coerce")
clv_table["last_purchase"]  = pd.to_datetime(clv_table["last_purchase"],  utc=True, errors="coerce")
clv_table["lifetime_days"]  = (clv_table["last_purchase"] - clv_table["first_purchase"]).dt.days

lt_all     = clv_table["lifetime_days"]
lt_repeat  = lt_all[lt_all > 0]          # exclude one-timers
mean_all   = lt_all.mean()
mean_rep   = lt_repeat.mean()
pct_zero   = (lt_all == 0).mean() * 100  # ~63%

fig, axes = plt.subplots(2, 1, figsize=(9, 10))


# ── Left panel: full distribution ────────────────────────────────────────────
ax = axes[0]
ax.hist(lt_all, bins=range(0, int(lt_all.max()) + 30, 30),
        color=BLUE, edgecolor="white", linewidth=0.3)
ax.axvline(mean_all, color=RED, linewidth=1.8, linestyle="--",
           label=f"Mean: {mean_all:.0f} days")
ax.annotate(
    f"{pct_zero:.1f}% of customers\nhave lifetime = 0\n(one-timers)",
    xy=(0, (lt_all == 0).sum()), xytext=(80, (lt_all == 0).sum() * 0.75),
    arrowprops=dict(arrowstyle="->", color="#555555"),
    fontsize=9, color="#555555"
)
ax.set_xlabel("Customer lifetime (days)")
ax.set_ylabel("Number of customers")
ax.set_title("All customers")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.legend(frameon=False)

# ── Right panel: repeat buyers only ──────────────────────────────────────────
ax = axes[1]
ax.hist(lt_repeat, bins=range(0, int(lt_repeat.max()) + 30, 30),
        color=BLUE, edgecolor="white", linewidth=0.3)
ax.axvline(mean_rep, color=RED, linewidth=1.8, linestyle="--",
           label=f"Mean: {mean_rep:.0f} days")
ax.set_xlabel("Customer lifetime (days)")
ax.set_ylabel("Number of customers")
ax.set_title(f"Repeat buyers only (n={len(lt_repeat):,})")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.legend(frameon=False)

fig.suptitle("Figure 3: Distribution of Historical Customer Lifetime", fontsize=13, fontweight="bold")
fig.tight_layout()
save("figure_3_historical_lifetime.png")


findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberati

Saved: figures\figure_3_historical_lifetime.png


## Figure 4: Predicted 6-Month CLV Distribution

In [23]:
clv      = lifetimes_summary["CLV_6m"]
cap      = clv.quantile(0.99)
mean_clv = clv.mean()

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(clv[clv <= cap], bins=60, color=BLUE, edgecolor="white", linewidth=0.3)
ax.axvline(mean_clv, color=RED, linewidth=1.8, linestyle="--", label=f"Mean: €{mean_clv:.0f}")
ax.set_xlabel("Predicted 6-month CLV per customer (€, top 1% trimmed)")
ax.set_ylabel("Number of customers")
ax.set_title("Figure 4: Distribution of Predicted 6-Month CLV")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"€{int(x):,}"))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.legend(frameon=False)
fig.tight_layout()
save("figure_4_predicted_clv.png")


findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberati

Saved: figures\figure_4_predicted_clv.png


## Figure 5: Mean Predicted CLV by Channel

In [24]:
ch = (
    lifetimes_summary.groupby("Channel")["CLV_6m"]
    .mean()
    .sort_values()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.barh(ch["Channel"], ch["CLV_6m"], color=BLUE, edgecolor="white", height=0.6)
for bar, val in zip(bars, ch["CLV_6m"]):
    ax.text(val + 1.5, bar.get_y() + bar.get_height() / 2,
            f"€{val:.0f}", va="center", fontsize=10)
ax.set_xlabel("Mean predicted 6-month CLV (€)")
ax.set_title("Figure 5: Mean Predicted CLV by Ordering Channel")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"€{int(x)}"))
ax.set_xlim(0, ch["CLV_6m"].max() * 1.18)
fig.tight_layout()
save("figure_5_clv_by_channel.png")


findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberati

Saved: figures\figure_5_clv_by_channel.png


## Figure 6: Mean Predicted CLV — Top 10 and Bottom 10 Stores

In [25]:
store_clv = (
    lifetimes_summary.groupby("StoreName")["CLV_6m"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
store_clv["label"] = store_clv["StoreName"].str.replace("Spare Rib Express ", "", regex=False)

top10    = store_clv.head(10).sort_values("CLV_6m")
bottom10 = store_clv.tail(10).sort_values("CLV_6m")

fig, axes = plt.subplots(2, 1, figsize=(9, 11))

for ax, data, title, color in [
    (axes[0], top10,    "Figure 6A: Top 10 Stores",    BLUE),
    (axes[1], bottom10, "Figure 6B: Bottom 10 Stores", ORANGE),
]:
    bars = ax.barh(data["label"], data["CLV_6m"], color=color, edgecolor="white", height=0.65)
    for bar, val in zip(bars, data["CLV_6m"]):
        ax.text(val + 1, bar.get_y() + bar.get_height() / 2,
                f"€{val:.0f}", va="center", fontsize=9)
    ax.set_xlabel("Mean predicted 6-month CLV (€)")
    ax.set_title(title, pad=8)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"€{int(x)}"))
    ax.set_xlim(0, data["CLV_6m"].max() * 1.22)

fig.suptitle("Mean Predicted CLV by Store", fontsize=13, fontweight="bold")
fig.tight_layout()
save("figure_6_clv_top_bottom_stores.png")


findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberati

Saved: figures\figure_6_clv_top_bottom_stores.png


## Figure R1: Revenue per Order — Stars vs Casuals (Owned Channels)

Segment labels are derived inline using the same logic as `Retention.ipynb`.

In [26]:
# Segment assignment (mirrors Retention.ipynb)
_repeaters = lifetimes_summary[lifetimes_summary["frequency"] > 0]
_clv_75th  = _repeaters["CLV_6m"].quantile(0.75)

def _assign_segment(row):
    if row["frequency"] == 0:      return "One-timer"
    elif row["CLV_6m"] >= _clv_75th: return "Star"
    else:                           return "Casual"

lifetimes_summary["segment"] = lifetimes_summary.apply(_assign_segment, axis=1)

owned  = ["Website", "ConsumerApp"]
stars  = lifetimes_summary[
    (lifetimes_summary["segment"] == "Star") &
    (lifetimes_summary["Channel"].isin(owned)) &
    (lifetimes_summary["monetary_value"] > 0)
]
casuals = lifetimes_summary[
    (lifetimes_summary["segment"] == "Casual") &
    (lifetimes_summary["Channel"].isin(owned)) &
    (lifetimes_summary["monetary_value"] > 0)
]

cap  = max(stars["monetary_value"].quantile(0.99), casuals["monetary_value"].quantile(0.99))
bins = np.linspace(0, cap, 55)

fig, axes = plt.subplots(2, 1, figsize=(9, 10))

for ax, data, label, color in [
    (axes[0], stars,   f"Stars  (n={len(stars):,})",   BLUE),
    (axes[1], casuals, f"Casuals (n={len(casuals):,})", ORANGE),
]:
    mv = data["monetary_value"].clip(upper=cap)
    ax.hist(mv, bins=bins, color=color, edgecolor="white", linewidth=0.3, alpha=0.85)
    ax.axvline(data["monetary_value"].mean(), color=RED, linewidth=1.8, linestyle="--",
               label=f"Mean: €{data['monetary_value'].mean():.0f}")
    ax.set_xlabel("Avg revenue per order (€)")
    ax.set_ylabel("Number of customers")
    ax.set_title(label)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"€{int(x)}"))
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
    ax.legend(frameon=False)

fig.suptitle("Figure R1: Revenue per Order — Stars vs Casuals (Owned Channels)",
             fontsize=13, fontweight="bold")
fig.tight_layout()
save("figure_R1_star_casual_order_value.png")


findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberati

Saved: figures\figure_R1_star_casual_order_value.png
